# https://qiskit-community.github.io/qiskit-nature/tutorials/06_qubit_mappers.html

1. Electronic structure Hamiltonian of the H2 molecule

In [1]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit

# driver = PySCFDriver()
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)
problem = driver.run()
fermionic_op = problem.hamiltonian.second_q_op()
print(fermionic_op)

Fermionic Operator
number spin orbitals=4, number terms=36
  0.3378550774017582 * ( +_0 +_0 -_0 -_0 )
+ 0.3322908651276483 * ( +_0 +_1 -_1 -_0 )
+ 0.3378550774017582 * ( +_0 +_2 -_2 -_0 )
+ 0.3322908651276483 * ( +_0 +_3 -_3 -_0 )
+ 0.09046559989211571 * ( +_0 +_0 -_1 -_1 )
+ 0.09046559989211571 * ( +_0 +_1 -_0 -_1 )
+ 0.09046559989211571 * ( +_0 +_2 -_3 -_1 )
+ 0.09046559989211571 * ( +_0 +_3 -_2 -_1 )
+ 0.09046559989211571 * ( +_1 +_0 -_1 -_0 )
+ 0.09046559989211571 * ( +_1 +_1 -_0 -_0 )
+ 0.09046559989211571 * ( +_1 +_2 -_3 -_0 )
+ 0.09046559989211571 * ( +_1 +_3 -_2 -_0 )
+ 0.3322908651276483 * ( +_1 +_0 -_0 -_1 )
+ 0.3492868613660083 * ( +_1 +_1 -_1 -_1 )
+ 0.3322908651276483 * ( +_1 +_2 -_2 -_1 )
+ 0.3492868613660083 * ( +_1 +_3 -_3 -_1 )
+ 0.3378550774017582 * ( +_2 +_0 -_0 -_2 )
+ 0.3322908651276483 * ( +_2 +_1 -_1 -_2 )
+ 0.3378550774017582 * ( +_2 +_2 -_2 -_2 )
+ 0.3322908651276483 * ( +_2 +_3 -_3 -_2 )
+ 0.09046559989211571 * ( +_2 +_0 -_1 -_3 )
+ 0.09046559989211571 * ( +_2

In [2]:
problem.hamiltonian.nuclear_repulsion_energy  # NOT included in the second_q_op above

np.float64(0.7199689944489797)

In [3]:
problem.molecule

MoleculeInfo(symbols=['H', 'H'], coords=[(0.0, 0.0, 0.0), (0.0, 0.0, 1.3889487015553204)], multiplicity=1, charge=0, units=<DistanceUnit.BOHR: 'Bohr'>, masses=[np.int64(1), np.int64(1)])

In [4]:
problem.reference_energy

np.float64(-1.1169989967540044)

In [5]:
problem.num_particles

(1, 1)

In [6]:
problem.num_spatial_orbitals

2

In [7]:
problem.basis

<ElectronicBasis.MO: 'molecular'>

2. The Jordan-Wigner Mapping


In [8]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()

In [9]:
qubit_jw_op = mapper.map(fermionic_op)
print(qubit_jw_op)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81054798+0.j,  0.17218393+0.j, -0.22575349+0.j,  0.12091263+0.j,
  0.17218393+0.j,  0.16892754+0.j, -0.22575349+0.j,  0.16614543+0.j,
  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,
  0.16614543+0.j,  0.17464343+0.j,  0.12091263+0.j])


In [10]:
### add nuclear energy to hamiltonian:

from qiskit.quantum_info import SparsePauliOp

E_nuc = problem.nuclear_repulsion_energy
print(f"E_nuc: {E_nuc}")

Hq = qubit_jw_op  # from mapper.map(...)
nq = Hq.num_qubits
print(f"num of qubits:  {nq}")

I = SparsePauliOp("I" * nq, coeffs=[E_nuc])
print(f"E_nuc I: \n{I}")

H_total = Hq + I
print(f"H_total: \n{H_total}")


E_nuc: 0.7199689944489797
num of qubits:  4
E_nuc I: 
SparsePauliOp(['IIII'],
              coeffs=[0.71996899+0.j])
H_total: 
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII', 'IIII'],
              coeffs=[-0.81054798+0.j,  0.17218393+0.j, -0.22575349+0.j,  0.12091263+0.j,
  0.17218393+0.j,  0.16892754+0.j, -0.22575349+0.j,  0.16614543+0.j,
  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,
  0.16614543+0.j,  0.17464343+0.j,  0.12091263+0.j,  0.71996899+0.j])


# Get Hamiltonian restricted to single-particle states
single_particle_H = np.zeros((n_qubits, n_qubits))
for i in range(n_qubits):
    for j in range(i + 1):
        for p, coeff in H_op.to_list():
            p_x = Pauli(p).x
            p_z = Pauli(p).z
            if all(p_x[k] == ((i == k) + (j == k)) % 2 for k in range(n_qubits)):
                sgn = ((-1j) ** sum(p_z[k] and p_x[k] for k in range(n_qubits))) * (
                    (-1) ** p_z[i]
                )
            else:
                sgn = 0
            single_particle_H[i, j] += sgn * coeff
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        single_particle_H[i, j] = np.conj(single_particle_H[j, i])

print(single_particle_H)
# Set dt according to spectral norm
dt = np.pi / np.linalg.norm(single_particle_H, ord=2)
dt3. The Parity Mapping

In [11]:
from qiskit_nature.second_q.mappers import ParityMapper

mapper = ParityMapper()

In [12]:
qubit_p_op = mapper.map(fermionic_op)
print(qubit_p_op)

SparsePauliOp(['IIII', 'IIIZ', 'IIZZ', 'IIZI', 'IZZI', 'IZZZ', 'ZZII', 'ZZIZ', 'ZXIX', 'IXZX', 'ZXZX', 'IXIX', 'IZIZ', 'ZZZZ', 'ZIZI'],
              coeffs=[-0.81054798+0.j,  0.17218393+0.j, -0.22575349+0.j,  0.12091263+0.j,
  0.17218393+0.j,  0.16892754+0.j, -0.22575349+0.j,  0.16614543+0.j,
  0.0452328 +0.j, -0.0452328 +0.j, -0.0452328 +0.j,  0.0452328 +0.j,
  0.16614543+0.j,  0.17464343+0.j,  0.12091263+0.j])


In [13]:
mapper = ParityMapper(num_particles=problem.num_particles)

In [14]:
qubit_op = mapper.map(fermionic_op)
print(qubit_op)

SparsePauliOp(['II', 'IZ', 'ZI', 'ZZ', 'XX'],
              coeffs=[-1.05237325+0.j,  0.39793742+0.j, -0.39793742+0.j, -0.0112801 +0.j,
  0.1809312 +0.j])


4. More advanced qubit reductions: Z2 symmetries in the Hilbert space of the qubit

In [15]:
tapered_mapper = problem.get_tapered_mapper(mapper)
print(type(tapered_mapper))

<class 'qiskit_nature.second_q.mappers.tapered_qubit_mapper.TaperedQubitMapper'>


In [16]:
qubit_op = tapered_mapper.map(fermionic_op)
print(qubit_op)

# H2 molecule is such a simple system that we can simulate it entirely on a single qubit!

SparsePauliOp(['I', 'Z', 'X'],
              coeffs=[-1.04109314+0.j, -0.79587485+0.j, -0.1809312 +0.j])


5. Interleaved ordering

In [17]:
from qiskit_nature.second_q.circuit.library import HartreeFock
hf_state = HartreeFock(2, (1, 1), JordanWignerMapper())
hf_state.draw()

┌───┐
q_0: ┤ X ├
     └───┘
q_1: ─────
     ┌───┐
q_2: ┤ X ├
     └───┘
q_3: ─────

In [18]:
from qiskit_nature.second_q.mappers import InterleavedQubitMapper
interleaved_mapper = InterleavedQubitMapper(JordanWignerMapper())
hf_state = HartreeFock(2, (1, 1), interleaved_mapper)
hf_state.draw()


┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ X ├
     └───┘
q_2: ─────
          
q_3: ─────

### diff tutorial https://quantum.cloud.ibm.com/learning/en/courses/quantum-chem-with-vqe/hamiltonian-construction

In [46]:
import numpy as np
from pyscf import ao2mo, gto, mcscf, scf

In [47]:
# 1. Define your molecule  H2
distance = 0.735
a = distance / 2
mol = gto.Mole()
mol.build(
    verbose=0,
    atom=[
        ["H", (0, 0, -a)],
        ["H", (0, 0, a)],
    ],
    basis="sto-6g",
    spin=0,
    charge=0,
    symmetry="Dooh",
)

In [48]:
mf = scf.RHF(mol)
mf.scf()

print(
    mf.energy_nuc(),
    mf.energy_elec()[0],
    mf.energy_tot(),
    mf.energy_tot() - mol.energy_nuc(),
)

0.7199689944489797 -1.8455976628764188 -1.125628668427439 -1.8455976628764188


In [49]:
active_space = range(mol.nelectron // 2 - 1, mol.nelectron // 2 + 1)

In [50]:
# 2. Generate fermionic Hamiltonian
E1 = mf.kernel()
mx = mcscf.CASCI(mf, ncas=2, nelecas=(1, 1))
mo = mx.sort_mo(active_space, base=0)
E2 = mx.kernel(mo)[:2]

In [51]:
h1e, ecore = mx.get_h1eff()
h2e = ao2mo.restore(1, mx.get_h2eff(), mx.ncas)

In [52]:
# 3. mapping to hamiltonian
from common.hamiltonians import build_hamiltonian

In [53]:
H = build_hamiltonian(ecore, h1e, h2e)
print(H)

accuracy of Cholesky decomposition  1.8923604151236598e-16
SparsePauliOp(['IIII', 'IIIZ', 'IZII', 'IIZI', 'ZIII', 'IZIZ', 'IIZZ', 'ZIIZ', 'IZZI', 'ZZII', 'ZIZI', 'YYYY', 'XXYY', 'YYXX', 'XXXX'],
              coeffs=[-0.09820182+0.j,  0.1740751 +0.j,  0.1740751 +0.j, -0.2242933 +0.j,
 -0.2242933 +0.j,  0.16891402+0.j,  0.1210099 +0.j,  0.16631441+0.j,
  0.16631441+0.j,  0.1210099 +0.j,  0.17504456+0.j,  0.04530451+0.j,
  0.04530451+0.j,  0.04530451+0.j,  0.04530451+0.j])


In [43]:
#LiH
distance = 1.56
mol = gto.Mole()
mol.build(
    verbose=0,
    atom=[["Li", (0, 0, 0)], ["H", (0, 0, distance)]],
    basis="sto-6g",
    spin=0,
    charge=0,
    symmetry="Coov",
)
mf = scf.RHF(mol)
E1 = mf.kernel()

# %% ----------------------------------------------------------------------------------------------

mx = mcscf.CASCI(mf, ncas=5, nelecas=(1, 1))
cas_space_symmetry = {"A1": 3, "E1x": 1, "E1y": 1}
mo = mcscf.sort_mo_by_irrep(mx, mf.mo_coeff, cas_space_symmetry)
E2 = mx.kernel(mo)[:2]
h1e, ecore = mx.get_h1eff()
h2e = ao2mo.restore(1, mx.get_h2eff(), mx.ncas)

In [44]:
H_lih = build_hamiltonian(ecore, h1e, h2e)

accuracy of Cholesky decomposition  3.3306690738754696e-16


In [45]:
print(H_lih)

SparsePauliOp(['IIIIIIIIII', 'IIIIIIIIIZ', 'IIIIZIIIII', 'IIIIIIIIYY', 'IIIIIIIIXX', 'IIIYYIIIII', 'IIIXXIIIII', 'IIIIIYZZZY', 'IIIIIXZZZX', 'YZZZYIIIII', 'XZZZXIIIII', 'IIIIIIIIZI', 'IIIZIIIIII', 'IIIIIYZZYI', 'IIIIIXZZXI', 'YZZYIIIIII', 'XZZXIIIIII', 'IIIIIIIZII', 'IIZIIIIIII', 'IIIIIIZIII', 'IZIIIIIIII', 'IIIIIZIIII', 'ZIIIIIIIII', 'IIIIZIIIIZ', 'IIIYYIIIIZ', 'IIIXXIIIIZ', 'YZZZYIIIIZ', 'XZZZXIIIIZ', 'IIIIIIIIZZ', 'IIIZIIIIIZ', 'IIIIIYZZYZ', 'IIIIIXZZXZ', 'YZZYIIIIIZ', 'XZZXIIIIIZ', 'IIIIIIIZIZ', 'IIZIIIIIIZ', 'IIIIIIZIIZ', 'IZIIIIIIIZ', 'IIIIIZIIIZ', 'ZIIIIIIIIZ', 'IIIIZIIIYY', 'IIIIZIIIXX', 'IIIIZYZZZY', 'IIIIZXZZZX', 'IIIIZIIIZI', 'IIIZZIIIII', 'IIIIZYZZYI', 'IIIIZXZZXI', 'YZZYZIIIII', 'XZZXZIIIII', 'IIIIZIIZII', 'IIZIZIIIII', 'IIIIZIZIII', 'IZIIZIIIII', 'IIIIZZIIII', 'ZIIIZIIIII', 'IIIIIYZZIY', 'IIIYYIIIYY', 'IIIXXIIIYY', 'YZZZYIIIYY', 'XZZZXIIIYY', 'IIIZIIIIYY', 'YZZYIIIIYY', 'XZZXIIIIYY', 'IIIIIIIZYY', 'IIZIIIIIYY', 'IIIIIIZIYY', 'IZIIIIIIYY', 'IIIIIZIIYY', 'ZIIIIIIIYY', 'IIIY

# !!!

In [1]:
from common.hamiltonians import build_H2_hamiltonian, build_LiH_hamiltonian

In [2]:
H2 = build_H2_hamiltonian()  # this includes the nuclear energy
print(H2)

0.7199689944489797 -1.8455976628764188 -1.125628668427439 -1.8455976628764188
accuracy of Cholesky decomposition  1.8923604151236598e-16
SparsePauliOp(['IIII', 'IIIZ', 'IZII', 'IIZI', 'ZIII', 'IZIZ', 'IIZZ', 'ZIIZ', 'IZZI', 'ZZII', 'ZIZI', 'YYYY', 'XXYY', 'YYXX', 'XXXX'],
              coeffs=[-0.09820182+0.j,  0.1740751 +0.j,  0.1740751 +0.j, -0.2242933 +0.j,
 -0.2242933 +0.j,  0.16891402+0.j,  0.1210099 +0.j,  0.16631441+0.j,
  0.16631441+0.j,  0.1210099 +0.j,  0.17504456+0.j,  0.04530451+0.j,
  0.04530451+0.j,  0.04530451+0.j,  0.04530451+0.j])


In [3]:
LiH = build_LiH_hamiltonian()
print(LiH)

accuracy of Cholesky decomposition  3.3306690738754696e-16
SparsePauliOp(['IIIIIIIIII', 'IIIIIIIIIZ', 'IIIIZIIIII', 'IIIIIIIIYY', 'IIIIIIIIXX', 'IIIYYIIIII', 'IIIXXIIIII', 'IIIIIYZZZY', 'IIIIIXZZZX', 'YZZZYIIIII', 'XZZZXIIIII', 'IIIIIIIIZI', 'IIIZIIIIII', 'IIIIIYZZYI', 'IIIIIXZZXI', 'YZZYIIIIII', 'XZZXIIIIII', 'IIIIIIIZII', 'IIZIIIIIII', 'IIIIIIZIII', 'IZIIIIIIII', 'IIIIIZIIII', 'ZIIIIIIIII', 'IIIIZIIIIZ', 'IIIYYIIIIZ', 'IIIXXIIIIZ', 'YZZZYIIIIZ', 'XZZZXIIIIZ', 'IIIIIIIIZZ', 'IIIZIIIIIZ', 'IIIIIYZZYZ', 'IIIIIXZZXZ', 'YZZYIIIIIZ', 'XZZXIIIIIZ', 'IIIIIIIZIZ', 'IIZIIIIIIZ', 'IIIIIIZIIZ', 'IZIIIIIIIZ', 'IIIIIZIIIZ', 'ZIIIIIIIIZ', 'IIIIZIIIYY', 'IIIIZIIIXX', 'IIIIZYZZZY', 'IIIIZXZZZX', 'IIIIZIIIZI', 'IIIZZIIIII', 'IIIIZYZZYI', 'IIIIZXZZXI', 'YZZYZIIIII', 'XZZXZIIIII', 'IIIIZIIZII', 'IIZIZIIIII', 'IIIIZIZIII', 'IZIIZIIIII', 'IIIIZZIIII', 'ZIIIZIIIII', 'IIIIIYZZIY', 'IIIYYIIIYY', 'IIIXXIIIYY', 'YZZZYIIIYY', 'XZZZXIIIYY', 'IIIZIIIIYY', 'YZZYIIIIYY', 'XZZXIIIIYY', 'IIIIIIIZYY', 'IIZIIIIIYY', 'I

/Users/justynapokora/PythonProjects/QuantumSubspaceExpansion/.venv/lib/python3.12/site-packages/pyscf/scf/addons.py:600: RuntimeWarning: divide by zero encountered in dot
  return lib.cho_solve(s22, numpy.dot(s21, mo1), strict_sym_pos=False)
/Users/justynapokora/PythonProjects/QuantumSubspaceExpansion/.venv/lib/python3.12/site-packages/pyscf/scf/addons.py:600: RuntimeWarning: overflow encountered in dot
  return lib.cho_solve(s22, numpy.dot(s21, mo1), strict_sym_pos=False)
/Users/justynapokora/PythonProjects/QuantumSubspaceExpansion/.venv/lib/python3.12/site-packages/pyscf/scf/addons.py:600: RuntimeWarning: invalid value encountered in dot
  return lib.cho_solve(s22, numpy.dot(s21, mo1), strict_sym_pos=False)
